# QG testbed — realism scenarios: when does 4DVar lose its advantage?

`qg2l_pipeline.ipynb` is a favourable case for 4DVar: same equations / parameters / forcing for the truth and the
DA model (truncation-only model error), observations without representativeness error, exactly known R, B estimated
from the truth, perfect H, periodic domain, unlimited optimisation budget. This notebook switches these
assumptions off one at a time (and all together), and compares for each scenario

* **4DVarNet** (retrained on each new dataset),
* **4DVar strong / weak** constraint (L-BFGS on the B^{1/2}-preconditioned cost, near convergence),
* **incremental 4DVar** as run operationally (Gauss-Newton outer loops + preconditioned conjugate gradient with
  tangent-linear / adjoint models), at an operational-like budget (3 x 20) and at a **low budget** (1 x 5),
* the **hybrid 4DVarNet → 4DVar** (4DVarNet reconstruction as background of each window),

with the usual scores and the **error energy per Lyapunov subspace** (unstable / neutral / stable / residual).
Expected picture: 4DVarNet's advantage on the unstable subspace persists, while 4DVar's advantage on the stable
subspace (dynamics for free) shrinks as model and observation errors become realistic.

Everything is cached by configuration hash (`contrib/qg2l/pipeline.py`): scenarios that share a dataset share the
trained 4DVarNet, and re-running only recomputes what changed. Each new dataset = one 4DVarNet training
(tens of minutes on a GPU in the small size), so choose the scenarios to run in `RUN`.

In [ ]:
import os, sys, json, copy
from pathlib import Path

REPO = Path.cwd().resolve()
while not (REPO / "main.py").exists():
    REPO = REPO.parent
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import Image, Video, display

from contrib.qg2l import pipeline, obs as O
from contrib.qg2l.simulate import PRESETS

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 0. Base configuration (same small set-up as `qg2l_pipeline.ipynb`)

In [ ]:
SIZE = "small"   # "small" | "full"
ROOT = Path("data/qg2l_scenarios_" + SIZE)
PRESET = "energetic"

if SIZE == "small":
    L, NX, NX_TRUTH, SPINUP = 5e5, 32, 64, 300
    SPLITS = dict(train=[0, 1095], val=[1095, 1185], test=[1215, 1305], total=1395)
    NET = dict(n_time=21, epochs=120)
    LYAP = dict(k=192, n_fwd=30, n_bwd=60, animation_days=30)
    WINDOW, SPIN_VAR = 7, 30
else:
    L, NX, NX_TRUTH, SPINUP = 1e6, 64, 128, 1095
    SPLITS = dict(train=[0, 1095], val=[1095, 1277], test=[1307, 1672], total=1762)
    NET = dict(n_time=21, epochs=150)
    LYAP = dict(k=512, n_fwd=30, n_bwd=60, animation_days=60)
    WINDOW, SPIN_VAR = 10, 60

# constellation with the real ground-track spacing whatever the box size
CONST = {n: dict(s, tracks_per_dir=max(1, round(s["tracks_per_dir"] * L / 1e6))) for n, s in O.DEFAULT_CONSTELLATION.items()}
BASE_GEN = dict(preset=PRESET, nx=NX, nx_truth=NX_TRUTH, L=L, spinup_days=SPINUP, splits=SPLITS, constellation=CONST,
                animation_days=30)

# DA methods compared in every scenario (any run_4dvar kwarg can be set; 'background': '4dvarnet' -> hybrid).
# All 4DVars use the B^{1/2} control-variable transform (preconditioning). Solvers:
#   lbfgs       : nonlinear L-BFGS on the preconditioned cost, run to (near) convergence -> reference
#   incremental : operational-style Gauss-Newton, n_outer re-linearisations x n_inner preconditioned CG
#                 iterations with tangent-linear / adjoint models
COMMON = dict(window_days=WINDOW, b_scale=0.5, spinup_days=SPIN_VAR)
METHODS = [
    dict(name="4dvar_strong", mode="strong", solver="lbfgs", n_iter=40, **COMMON),
    dict(name="4dvar_weak", mode="weak", solver="lbfgs", n_iter=40, q_scale=0.2, **COMMON),
    dict(name="4dvar_incr", mode="strong", solver="incremental", n_outer=3, n_inner=20, **COMMON),
    # low budget: 1 outer loop x 5 CG iterations ~ 6 (TL + AD) integrations per window
    dict(name="4dvar_incr_budget", mode="strong", solver="incremental", n_outer=1, n_inner=5, **COMMON),
    dict(name="hybrid", mode="strong", solver="incremental", n_outer=1, n_inner=10, window_days=WINDOW,
         b_scale=0.2, spinup_days=0, background="4dvarnet"),
]

## 1. Scenarios

| scenario | what changes | assumption broken |
|---|---|---|
| `baseline` | nothing | — |
| `obs_hr` | obs sample the block-averaged **high-resolution** SSH | no representativeness error |
| `obs_signal` | + an **unresolved SSH signal** (internal-tide-like, 1 cm, 40–120 km, 1-day decorrelation) in the obs only | obs = model-resolvable truth + white noise |
| `swot_roll` | SWOT **roll-like correlated error** (1 cm / 100 km), R still diagonal in 4DVar | R known and diagonal |
| `param_error` | truth with **different physics** (bottom drag ×1.5, β ×0.8) | perfect model equations |
| `forcing` | truth with an **unknown stochastic forcing** (2 mm/day, 150–400 km) | perfect model forcing |
| `B_free_run` | 4DVar **B from a free run of the DA model** instead of the truth | oracle B |
| `nadir_only` | no SWOT | (observability: larger λ₁·τ_obs) |
| `realistic` | everything above together (with SWOT) | all |

In [ ]:
P = PRESETS[PRESET]
SIGNAL = dict(ssh_std=0.01, wavelength_km=(40.0, 120.0), decorrelation_days=1.0)
CONST_ROLL = {n: dict(s, roll_std=0.01 if s["kind"] == "swot" else 0.0) for n, s in CONST.items()}
CONST_NADIR = {n: s for n, s in CONST.items() if s["kind"] == "nadir"}
PARAMS = dict(rek=1.5 * P["rek"], beta=0.8 * P["beta"])
FORCING = dict(ssh_std=0.002, wavelength_km=(150.0, 400.0))

SCENARIOS = {
    "baseline":    dict(gen={}),
    "obs_hr":      dict(gen=dict(obs_source="hr_block")),
    "obs_signal":  dict(gen=dict(obs_source="hr_block", unresolved_signal=SIGNAL)),
    "swot_roll":   dict(gen=dict(constellation=CONST_ROLL)),
    "param_error": dict(gen=dict(truth_overrides=PARAMS)),
    "forcing":     dict(gen=dict(forcing=FORCING)),
    "B_free_run":  dict(gen={}, var=dict(b_source="free_run")),
    "nadir_only":  dict(gen=dict(constellation=CONST_NADIR)),
    "realistic":   dict(gen=dict(obs_source="hr_block", unresolved_signal=SIGNAL, constellation=CONST_ROLL,
                                 truth_overrides=PARAMS, forcing=FORCING), var=dict(b_source="free_run")),
}

# scenarios to run now (each new dataset = one 4DVarNet training)
RUN = ["baseline", "obs_signal", "param_error", "B_free_run", "realistic"]

## 2. Run

**Following progress.** The cell below runs everything sequentially; while it runs, the kernel is busy, so follow it
from a terminal on the cluster (same folder, same conda env):

```bash
python -m contrib.qg2l.pipeline data/qg2l_scenarios_small         # one-shot status
watch -n 60 python -m contrib.qg2l.pipeline data/qg2l_scenarios_small
tail -f data/qg2l_scenarios_small/4dvarnet/*/logs/version_*/metrics.csv   # per-epoch metrics
```

In [ ]:
RESULTS = {}
for name in RUN:
    sc = SCENARIOS[name]
    gen = {**BASE_GEN, **sc.get("gen", {})}
    methods = [{**m, **sc.get("var", {})} for m in METHODS]
    print(f"\n========== {name} ==========")
    try:
        RESULTS[name] = pipeline.run_scenario(ROOT, name, gen, methods, net=NET, lyap=LYAP, device=DEVICE)
    except Exception as e:  # keep going with the other scenarios
        import traceback; traceback.print_exc()
        print(f"!!! scenario {name} failed: {e}")

In [ ]:
pipeline.progress(ROOT)   # state of datasets / trainings / scenarios (also after a restart)

## 3. Observation statistics and dynamical regime per scenario

In [ ]:
pd.DataFrame({n: {k: r["obs_summary"][k] for k in ["lambda1_per_day_LRmodel", "median_revisit_days",
                                                   "lambda1_x_revisit", "ssh_std_m", "obs_minus_truth_std_m"]}
              for n, r in RESULTS.items()}).T.round(4)

## 4. Scores (SSH and deep layer)

In [ ]:
scores = pd.concat({n: pd.DataFrame(r["scores"]).T for n, r in RESULTS.items()}, names=["scenario", "method"])
scores.round(4)

In [ ]:
# number of model integrations (nonlinear / tangent linear / adjoint) over the whole test run
pd.concat({n: pd.DataFrame(r["cost"]).T for n, r in RESULTS.items() if r.get("cost")}, names=["scenario", "method"])

In [ ]:
METHOD_ORDER = ["4dvarnet", "4dvar_strong", "4dvar_weak", "4dvar_incr", "4dvar_incr_budget", "hybrid"]
fig, ax = plt.subplots(1, 4, figsize=(18, 3.8), constrained_layout=True)
for a, key in zip(ax, ["rmse_ssh", "lx_ssh_km", "rmse_layer2", "lx_layer2_km"]):
    tab = scores[key].unstack("method").reindex(columns=[m for m in METHOD_ORDER if m in scores.index.get_level_values(1)])
    tab.loc[list(RESULTS)].plot.bar(ax=a, legend=a is ax[0], rot=30)
    a.set_title(key)

## 5. Error energy per Lyapunov subspace

In [ ]:
SUBS = ["blv_unstable_abs", "blv_neutral_abs", "blv_stable_resolved_abs", "residual_abs"]
rows = []
for n, r in RESULTS.items():
    if r["lyap"] is None:
        continue
    for m, v in r["lyap"]["methods"].items():
        rows.append(dict(scenario=n, method=m, n_unstable=r["lyap"]["n_unstable"],
                         total=v["mean_error_energy"], **{s: v["mean_" + s] for s in SUBS},
                         angle_us_deg=v.get("mean_angle_us_deg", np.nan)))
lyap_tab = pd.DataFrame(rows).set_index(["scenario", "method"])
lyap_tab

In [ ]:
fig, ax = plt.subplots(1, len(SUBS) + 1, figsize=(4.2 * (len(SUBS) + 1), 4), constrained_layout=True, sharey=False)
for a, key in zip(ax, ["total"] + SUBS):
    tab = lyap_tab[key].unstack("method")
    tab = tab.reindex(columns=[m for m in METHOD_ORDER if m in tab.columns]).loc[[n for n in RESULTS if n in tab.index]]
    tab.plot(ax=a, marker="o", logy=True, legend=a is ax[0], rot=30)
    a.set_title(key.replace("_abs", "").replace("blv_", ""))
    a.set_xticks(range(len(tab))); a.set_xticklabels(tab.index, rotation=30)
fig.suptitle("mean error energy per subspace (energy norm), by scenario");

In [ ]:
# relative gain of 4DVarNet over the strong-constraint 4DVar per subspace: > 0 means 4DVarNet is better
gain = (1 - lyap_tab.xs("4dvarnet", level="method")[["total"] + SUBS]
        / lyap_tab.xs("4dvar_strong", level="method")[["total"] + SUBS])
gain.round(3)

## 6. Detailed diagnostic of one scenario

In [ ]:
SHOW = "realistic" if "realistic" in RESULTS else list(RESULTS)[0]
d = RESULTS[SHOW]["diag_dir"]
for f in ["lyapunov_spectrum_error_distribution.png", "error_energy_by_subspace.png", "subspace_summary.png",
          "clv_and_error_maps.png"]:
    if (d / f).exists():
        display(Image(filename=str(d / f)))
anim = d / "reconstructions.mp4"
if anim.exists():
    display(Video(str(anim), embed=True, width=1100))

In [ ]:
# 4DVarNet training curves for that scenario
hist = pipeline.training_curves(RESULTS[SHOW]["recs"]["4dvarnet"])
if not hist.empty:
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5), constrained_layout=True)
    hist[[c for c in ["train_loss", "val_loss"] if c in hist]].dropna(how="all").plot(ax=ax[0], logy=True)
    hist[[c for c in ["train_mse", "val_mse"] if c in hist]].dropna(how="all").plot(ax=ax[1], logy=True)
    ax[0].set_title("loss"); ax[1].set_title("MSE")

## 7. Reading the results

* **unstable** column: controlled by observability + optimisation. A dynamically consistent estimate (4DVar) lets its
  error grow along these directions within each window (saw-tooth in `error_energy_by_subspace.png`); the learned
  smoother does not have to be a model trajectory.
* **stable / residual** columns: in the baseline, 4DVar gets them almost for free from the exact dynamics. Model
  error (`param_error`, `forcing`) and unresolved signal in the obs (`obs_signal`) should erode this advantage;
  if it persists, the learned prior is the bottleneck (try a richer prior / more data).
* **4dvar_incr vs 4dvar_strong**: operational incremental solver vs (near) converged nonlinear minimisation.
  **4dvar_incr_budget**: what is left of a properly preconditioned 4DVar at a few TL/AD integrations per window
  (compare with the cost of a 4DVarNet forward pass in the cost table).
* **hybrid**: if it beats both, the two error structures are complementary (4DVarNet fixes the unstable part, the
  short 4DVar re-imposes dynamical consistency on the stable part).
* Tune `b_scale` / `q_scale` / window on the validation split before final conclusions (`split="val"` in a method dict).